# Cross-Encoders and the Reranking Cascade

This notebook is the narrative companion to `cross_encoders_reranking.py`, which **owns every number**
the topic and its laboratory use. We never reimplement the mathematics here — we import the reference
module and walk the three movements section by section.

The dual-encoder topic kept invoking the cross-encoder as the *rank-free, un-precomputable counterpoint*.
Here we spend that power: a cross-encoder scores a **fused** pair $s(q,d) = h([q;d])$, abandoning the
separable inner product that made dual retrieval precomputable.


In [ ]:
import cross_encoders_reranking as ce

s = ce._setup()
nq, npass = s["S_full"].shape
print(f"finance corpus: {nq} queries, {npass} documents (one per company)")
print(f"first stage: a rank-{ce.D_STAGE1} dual encoder; recall@1 = "
      f"{ce.stage1_recall_at_k(s['order'], s['truth'], 1):.3f}")


## Movement 1 — joint encoding breaks the rank ceiling, and the nonlinearity is why

A dual encoder's score matrix $S = QG^\top$ has rank $\le d$. The subtle, load-bearing fact: a
**learned bilinear** scorer $q^\top W d$ does *not* escape it, because $S = QWG^\top = (QW)G^\top$ is a
product through a $d$-dimensional bottleneck — just a dual encoder with reparametrized queries. Only a
genuinely **nonlinear** joint scorer escapes.

We demonstrate it on the signed-identity target imported from the embedding-dimension topic (full rank
$n$). The best rank-$d$ dual encoder (= the best learned bilinear = truncated SVD) plateaus at a
positive reconstruction error for $d < n$; the random-ReLU cross-encoder reconstructs it to machine
precision at every $d$.


In [ ]:
M = ce.signed_identity(ce.SIGN_N)
curve = ce.recon_error_curve(M, ce.M1_DIMS, ce.CE_N_FEAT, ce.CE_SEED, ce.CE_GAMMA, ce.CE_LAM)
print(f"signed_identity({ce.SIGN_N}): rank {int(__import__('numpy').linalg.matrix_rank(M))}")
print("  d  | rank-d ceiling | cross-encoder")
for r in curve:
    print(f"  {r['d']}  |    {r['ceiling']:.3f}     |   {r['cross']:.1e}")


The learned bilinear is no help: with the SVD's singular vectors as embeddings and $W$ the diagonal
of singular values, $QWP^\top$ *is* the truncated SVD — so the best bilinear and the best dual encoder
are the same rank-$d$ ceiling. A learned interaction matrix changes nothing.


In [ ]:
import numpy as np
for d in (1, 3, 6):
    same = np.allclose(ce.best_bilinear_ceiling(M, d), ce.best_rank_d(M, d), atol=1e-9)
    rk = int(np.linalg.matrix_rank(ce.bilinear_score(np.random.default_rng(0).standard_normal((6, d)),
              np.random.default_rng(1).standard_normal((d, d)),
              np.random.default_rng(2).standard_normal((6, d)))))
    print(f"d={d}: best bilinear == truncated SVD? {same};  rank(arbitrary Q W P^T) = {rk} (<= {d})")


On the finance geometry the same cross-encoder $h([q;d])$ realizes the relevance **exactly** —
recall@1 $= 1.0$, separating the same-sector hard negatives the rank-3 dual first stage confuses.
(It is fit on the full relevance; the train/test coincidence is the expressivity-not-generalization
caveat in the rigorFlag.)


In [ ]:
Sce = ce.cross_encoder_finance_scores(s["Q"], s["P"], s["truth"])
print(f"cross-encoder recall@1 = {ce.topk_recall(Sce, s['truth'], 1):.3f}")
print(f"rank-{ce.D_STAGE1} dual stage recall@1 = {ce.stage1_recall_at_k(s['order'], s['truth'], 1):.3f}")


## Movement 2 — the retrieve-then-rerank cascade and the recall pinch

The cross-encoder is linear in what it scores, so it cannot score the whole corpus; it **reranks** a
cheap first stage's top-$K$. The cost is $c_{\text{ret}} + K\,c_{\text{ce}}$ against the brute
$|\mathcal{C}|\,c_{\text{ce}}$.

Under a known-item qrel (one gold document per query), an **oracle** rerank of the top-$K$ makes
recall@1 equal the candidate pool's recall@$K$ — it *lifts* recall@1 from the first stage's recall@1 up
to its recall@$K$ and *caps* it there. The cascade can never recover a gold the first stage dropped.


In [ ]:
rows = ce.cascade_recall_vs_k(s["order"], s["truth"], s["S_full"],
                              ce.LOSSY_SIGMA, ce.LOSSY_SEED, ce.K_GRID)
print("  K | stage-1 recall@K | oracle rerank r@1 | (decent) real rerank r@1")
for r in rows:
    print(f"  {r['K']} |      {r['stage1_rk']:.3f}      |      {r['oracle']:.3f}       |   {r['lossy']:.3f}")
print("\noracle rerank r@1 == stage-1 recall@K at every K (the recall pinch).")


## Movement 3 — monotonicity, and when reranking hurts

An **oracle** rerank is recall-monotone in $K$ (a larger top-$K$ is a superset). A **lossy**
cross-encoder can *dip* recall@1 below the first stage by being confidently wrong — and, the sharpest
contrast, more over-fetch makes the dip *worse*. We sweep the reranker quality (the corruption $\sigma$,
$\sigma=0$ being the oracle) and bucket each query: `fixed` (a hard-negative win), `kept`, `broke`
(the confident-wrong dip), `missed`.


In [ ]:
vals = [ce.oracle_rerank_recall(s["order"], s["truth"], K) for K in ce.K_GRID]
print("oracle rerank recall@1 by K (monotone):", [round(v, 3) for v in vals])
print()
print(" sigma | fixed kept broke missed | net lift")
for g in ce.SIGMA_GRID:
    L = ce.lossy_scores(s["S_full"], g, ce.LOSSY_SEED)
    b = ce.rerank_buckets(s["order"], s["truth"], L, ce.DIP_K)
    print(f"  {g:.2f} |   {b['fixed']}    {b['kept']}    {b['broke']}     {b['missed']}   | {b['net_lift']:+.3f}")


**Closing remark — expressivity is paid for in samples.** The cross-encoder's capacity is exactly what
overfits a small training set where the dual encoder's inner-product inductive bias generalizes. We
state this and up-link it to learning theory (VC dimension / generalization bounds) rather than simulate
it here.

## Verification

Every pedagogical claim in the topic is an `assert` in the reference module. Running the harness is the
regression test for the whole topic.


In [ ]:
ce._run_all()
print("\nall checks passed.")
